In [ ]:
# Initialization
import ee
import geemap

try: ee.Initialize(project='geeexercise')
except: ee.Authenticate(); ee.Initialize(project='geeexercise')

In [ ]:
# Configurations
aoi = ee.FeatureCollection('projects/geeexercise/assets/gebe-island-aoi-land').geometry()
crs, scale = 'EPSG:32752', 10
years = range(2019, 2026)
export_folder = 'gee-exports-new/phase-2'
asset_base = 'projects/geeexercise/assets/new-thesis-img'

print(f"Locked. AOI: {round(aoi.area(1).getInfo()/1e4, 1)} ha - Years: {list(years)}")

In [ ]:
# Load asset from previous stage
def pull_assets(yr):
    s2 = ee.Image(f'{asset_base}/S2_Composite_Gebe_{yr}').select([f'b{i+1}' for i in range(6)], ['B2','B3','B4','B8','B11','B12']).divide(10000)
    s1 = ee.Image(f'{asset_base}/S1_Composite_Gebe_{yr}').select(['b1','b2'], ['VV','VH'])
    return s2.clip(aoi), s1.clip(aoi)

In [ ]:
# Compute indices
def make_stack(s2, s1):
    EPS = ee.Image.constant(1e-10).toFloat()
    b = s2.select

    ndvi = b('B8').subtract(b('B4')).divide(b('B8').add(b('B4')).add(EPS))
    evi  = b('B8').subtract(b('B4')).divide(b('B8').add(b('B4').multiply(6)).subtract(b('B2').multiply(7.5)).add(1).add(EPS)).multiply(2.5)

    # MSAVI2
    two_nir = b('B8').multiply(2).add(1)
    radicand = two_nir.pow(2).subtract(b('B8').subtract(b('B4')).multiply(8)).max(0)
    msavi2 = two_nir.subtract(radicand.sqrt()).divide(2)

    bsi  = b('B11').add(b('B4')).subtract(b('B8').add(b('B2'))).divide(b('B11').add(b('B4')).add(b('B8')).add(b('B2')).add(EPS))
    ndwi = b('B3').subtract(b('B8')).divide(b('B3').add(b('B8')).add(EPS))
    nbr  = b('B8').subtract(b('B12')).divide(b('B8').add(b('B12')).add(EPS))
    clay = b('B11').divide(b('B12').add(EPS))
    fer  = b('B12').divide(b('B8').add(EPS))
    iron = b('B4').divide(b('B2').add(EPS))
    vvr  = s1.select('VV').subtract(s1.select('VH'))

    return ee.Image.cat([ndvi, evi, msavi2, bsi, ndwi, nbr, clay, fer, iron,
                         s1.select('VV'), s1.select('VH'), vvr]) \
           .rename(['NDVI','EVI','MSAVI2','BSI','NDWI','NBR','Clay','Ferrous','IronOxide','VV','VH','VV_VH_ratio']) \
           .toFloat()

In [ ]:
# Export
tasks = []
print("Building graphs and queueing exports...")

for yr in years:
    s2, s1 = pull_assets(yr)
    stack = make_stack(s2, s1)

    tasks.append(ee.batch.Export.image.toDrive(
        stack, f'FeatureStack_{yr}', export_folder,
        region=aoi, scale=scale, crs=crs, maxPixels=1e13, fileFormat='GeoTIFF'))

for t in tasks: t.start()
print(f"\nSubmitted {len(tasks)} tasks. Monitor: https://code.earthengine.google.com/tasks")

In [ ]:
# QA, here I only check for NDVI, BSI, and VV/VH ratio
lat, lon = aoi.centroid().coordinates().getInfo()[::-1]

Map = geemap.Map(center=[lat, lon], zoom=11)
Map.add_basemap('SATELLITE')

s2_24, s1_24 = pull_assets(2024)
stack_24 = make_stack(s2_24, s1_24)

Map.addLayer(stack_24.select('NDVI'), {'min':-0.2, 'max':0.9, 'palette':['brown','yellow','green']}, 'NDVI 2024')
Map.addLayer(stack_24.select('BSI'), {'min':-0.4, 'max':0.4, 'palette':['green','yellow','brown']}, 'BSI 2024', shown=False)
Map.addLayer(stack_24.select('VV_VH_ratio'), {'min':2, 'max':15, 'palette':['blue','white','red']}, 'VV/VH Ratio 2024', shown=False)
Map.addLayer(aoi, {'color':'white'}, 'AOI')
Map.addLayerControl()
Map